In [ ]:
_hook.remove()
print("Mid-layer hook removed.")

## Cleanup

Remove the forward hook when you're done to avoid interference with other code.

In [ ]:
threshold_configs = [
    (3.0, 0.1),   # Aggressive: accept more from CTC
    (1.0, 0.2),   # Balanced (default)
    (0.5, 0.3),   # Conservative: rely more on LLM verify
]

print("Threshold Experiment")
print("=" * 60)
for ctc_thresh, conf_thresh in threshold_configs:
    CTC_THRESHOLD        = ctc_thresh
    CONFIDENCE_THRESHOLD = conf_thresh
    preds, stats = speculative_transcribe([audio])
    print(f"\nCTC={ctc_thresh}, Conf={conf_thresh}:")
    print(f"  Time: {stats['total_time']*1000:.1f}ms  "
          f"Path: CTC={stats['ctc_accepted']}, LLM={stats['llm_accepted']}, AR={stats['fallback_count']}")
    print(f"  {preds[0][:120]}")

# Reset to defaults
CTC_THRESHOLD        = 0.7
CONFIDENCE_THRESHOLD = 0.2

## Tuning Thresholds

- **`CTC_THRESHOLD`**: max entropy for accepting the BPE draft directly (no LLM call). Higher = more CTC acceptances, faster but riskier.
- **`CONFIDENCE_THRESHOLD`**: minimum per-token LLM probability to accept the draft. Lower = more LLM acceptances, fewer AR fallbacks.

BPE entropy values are larger than grapheme entropy (bigger vocabulary), so `CTC_THRESHOLD` values in the 0.5–3.0 range are typical.

In [ ]:
@torch.no_grad()
def standard_transcribe(audios):
    """Standard AR decoding for comparison."""
    start_time   = time.time()
    texts        = [text_prompt] * len(audios)
    model_inputs = processor(texts, audios, device=device, return_tensors="pt").to(device)
    outputs      = model.generate(**model_inputs, max_new_tokens=MAX_NEW_TOKENS,
                                  num_beams=NUM_BEAMS, do_sample=False)
    predictions  = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
    return predictions, time.time() - start_time

print("Running comparison...\n")
ar_predictions, ar_time     = standard_transcribe([audio])
sp_predictions, sp_stats    = speculative_transcribe([audio])

print("COMPARISON")
print("=" * 50)
print(f"\nStandard AR:   {ar_time*1000:.1f}ms")
print(f"  {ar_predictions[0]}")
print(f"\nBPE Speculative: {sp_stats['total_time']*1000:.1f}ms  "
      f"(CTC={sp_stats['ctc_accepted']}, LLM={sp_stats['llm_accepted']}, AR={sp_stats['fallback_count']})")
print(f"  {sp_predictions[0]}")
print(f"\nSpeedup: {ar_time / sp_stats['total_time']:.2f}x")

## Comparison: BPE Speculative vs Standard AR

In [ ]:
predictions, stats = speculative_transcribe([audio])

print("=" * 50)
print("BPE SPECULATIVE DECODING RESULTS")
print("=" * 50)
print(f"\nTranscription: {predictions[0]}")
print(f"\nTiming:")
print(f"  CTC draft:     {stats['ctc_time']*1000:.1f}ms")
print(f"  LLM verify:    {stats['verify_time']*1000:.1f}ms")
print(f"  AR fallback:   {stats['fallback_time']*1000:.1f}ms")
print(f"  Total:         {stats['total_time']*1000:.1f}ms")
print(f"\nAcceptance:")
print(f"  CTC accepted:  {stats['ctc_accepted']}/{stats['batch_size']}")
print(f"  LLM accepted:  {stats['llm_accepted']}/{stats['batch_size']}")
print(f"  Fallback:      {stats['fallback_count']}/{stats['batch_size']}")

In [ ]:
import time

@torch.no_grad()
def speculative_transcribe(audios):
    """Complete BPE speculative decoding pipeline."""
    batch_sz   = len(audios)
    start_time = time.time()

    # Step 1: BPE CTC draft (single encoder pass; embeddings reused in steps 2-4)
    ctc_start = time.time()
    bpe_texts, bpe_entropies, embeddings, embed_lengths = ctc_decode_bpe(audios)
    ctc_time  = time.time() - ctc_start

    # Step 2: Gate by BPE CTC entropy
    predictions  = [None] * batch_sz
    verify_idx   = []
    ctc_accepted = 0

    for i, (text, ent) in enumerate(zip(bpe_texts, bpe_entropies)):
        if ent <= CTC_THRESHOLD and text.strip():
            predictions[i] = text.strip()
            ctc_accepted  += 1
        else:
            verify_idx.append(i)

    # Step 3: Verify remaining with LLM (reuses encoder embeddings)
    verify_time  = 0
    llm_accepted = 0

    if verify_idx:
        verify_start = time.time()
        results, audio_embeds, proj_lengths = verify(
            [bpe_texts[i] for i in verify_idx],
            embeddings[verify_idx],
            [embed_lengths[i] for i in verify_idx],
        )
        verify_time = time.time() - verify_start

        fail_idx = []
        for j, (accepted, text) in enumerate(results):
            i = verify_idx[j]
            if accepted:
                predictions[i] = text.strip()
                llm_accepted  += 1
            else:
                fail_idx.append(j)

        # Step 4: AR fallback
        if fail_idx:
            fallback_start = time.time()
            fallback_texts = fallback(audio_embeds, fail_idx, proj_lengths)
            fallback_time  = time.time() - fallback_start
            for k, j in enumerate(fail_idx):
                predictions[verify_idx[j]] = fallback_texts[k]
        else:
            fallback_time = 0
    else:
        fallback_time = 0

    total_time = time.time() - start_time

    stats = {
        'total_time':    total_time,
        'ctc_time':      ctc_time,
        'verify_time':   verify_time,
        'fallback_time': fallback_time,
        'ctc_accepted':  ctc_accepted,
        'llm_accepted':  llm_accepted,
        'fallback_count': batch_sz - ctc_accepted - llm_accepted,
        'batch_size':    batch_sz,
    }
    return predictions, stats

## Complete Pipeline

In [ ]:
fallback_result = fallback(audio_embeds, [0], proj_lengths)
print(f"Fallback result: {fallback_result[0]}")

In [ ]:
@torch.no_grad()
def fallback(audio_embeds, indices, proj_lengths):
    """AR fallback for samples that failed verification."""
    if not indices:
        return []

    batch_sz   = len(indices)
    hidden_dim = audio_embeds.shape[-1]
    all_embeds, all_lengths = [], []

    for i in indices:
        sample_embeds = audio_embeds[i, :proj_lengths[i], :].unsqueeze(0)
        combined = torch.cat([cached_prefix_embeds, sample_embeds, cached_suffix_embeds], dim=1)
        all_embeds.append(combined.squeeze(0))
        all_lengths.append(combined.shape[1])

    max_len   = max(all_lengths)
    padded    = torch.zeros(batch_sz, max_len, hidden_dim, device=device, dtype=audio_embeds.dtype)
    attn_mask = torch.zeros(batch_sz, max_len, dtype=torch.long, device=device)
    for i, (emb, length) in enumerate(zip(all_embeds, all_lengths)):
        padded[i, max_len - length:] = emb
        attn_mask[i, max_len - length:] = 1

    outputs = model.language_model.generate(
        inputs_embeds=padded, attention_mask=attn_mask,
        bos_token_id=tokenizer.bos_token_id, pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id, max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS, early_stopping=False, do_sample=False, use_cache=True
    )

    return [tokenizer.decode(outputs[i], skip_special_tokens=True) for i in range(batch_sz)]

## Step 4: Autoregressive Fallback

If the LLM rejects the draft, we fall back to standard greedy/beam decoding.
The `audio_embeds` already projected in the verify step are reused — no extra computation.

In [ ]:
results, audio_embeds, proj_lengths = verify(bpe_texts, embeddings, embed_lengths)

accepted, text = results[0]
print(f"Verification: {'ACCEPTED' if accepted else 'REJECTED'}")
print(f"Text: {text}")

In [ ]:
@torch.no_grad()
def verify(ctc_texts, embeddings, embed_lengths):
    """Verify BPE draft tokens with LLM in a single forward pass.

    Args:
        ctc_texts:    List of BPE-decoded draft strings
        embeddings:   Full-resolution enc_hidden [B, T_enc, D] from ctc_decode_bpe
        embed_lengths: Unpadded encoder frame count per sample

    Returns:
        results:      List of (accepted: bool, text: str)
        audio_embeds: Projected audio embeddings [B, T_proj, D] for fallback reuse
        proj_lengths: Projected length per sample
    """
    batch_sz = len(ctc_texts)

    ctc_token_ids = []
    for text in ctc_texts:
        text = text.strip() if text else ""
        ctc_token_ids.append(tokenizer.encode(text, add_special_tokens=False) if text else [])

    autocast = (torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
                if device.startswith('cuda') else contextlib.nullcontext())

    with autocast:
        audio_embeds = model.projector(embeddings)
    max_proj_len = audio_embeds.shape[1]

    window_size, downsample_rate = model.config.window_size, model.config.downsample_rate
    num_queries  = window_size // downsample_rate
    proj_lengths = [min(math.ceil(enc_len / window_size) * num_queries, max_proj_len)
                    for enc_len in embed_lengths]

    if not any(ctc_token_ids):
        return [(False, ctc_texts[i]) for i in range(batch_sz)], audio_embeds, proj_lengths

    audio_token_id                    = model.config.audio_token_id
    all_input_ids, prompt_lens, audio_ranges = [], [], []

    for i, proj_len in enumerate(proj_lengths):
        audio_start = len(prefix_ids)
        audio_ranges.append((audio_start, audio_start + proj_len))
        prompt_part = prefix_ids + [audio_token_id] * proj_len + suffix_ids
        prompt_lens.append(len(prompt_part))
        all_input_ids.append(prompt_part + ctc_token_ids[i])

    max_len    = max(len(ids) for ids in all_input_ids)
    padded_ids = torch.full((batch_sz, max_len), tokenizer.pad_token_id,
                             dtype=torch.long, device=device)
    attn_mask  = torch.zeros(batch_sz, max_len, dtype=torch.long, device=device)
    for i, ids in enumerate(all_input_ids):
        padded_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)
        attn_mask[i,  :len(ids)] = 1

    inputs_embeds = model.language_model.get_input_embeddings()(padded_ids)
    for i in range(batch_sz):
        s, e = audio_ranges[i]
        inputs_embeds[i, s:e, :] = audio_embeds[i, :e-s, :]

    with autocast:
        hidden = model.language_model.model(
            attention_mask=attn_mask, inputs_embeds=inputs_embeds, use_cache=False
        ).last_hidden_state

    # Gather hidden states at verification positions
    sample_idx, pos_idx, ctc_flat = [], [], []
    sample_ranges, sample_valid   = [], []
    offset = 0

    for i in range(batch_sz):
        ctc_tokens = ctc_token_ids[i]
        if not ctc_tokens or prompt_lens[i] - 1 + len(ctc_tokens) > hidden.shape[1]:
            sample_ranges.append((offset, offset))
            sample_valid.append(False)
            continue
        verify_start = prompt_lens[i] - 1
        for k in range(len(ctc_tokens)):
            sample_idx.append(i)
            pos_idx.append(verify_start + k)
            ctc_flat.append(ctc_tokens[k])
        sample_ranges.append((offset, offset + len(ctc_tokens)))
        sample_valid.append(True)
        offset += len(ctc_tokens)

    if pos_idx:
        gathered = hidden[torch.tensor(sample_idx, device=device),
                          torch.tensor(pos_idx,    device=device), :]
        with autocast:
            logits = model.language_model.lm_head(gathered) / logits_scaling
        probs     = F.softmax(logits.float(), dim=-1)
        ctc_probs = probs[torch.arange(len(ctc_flat), device=device),
                          torch.tensor(ctc_flat, device=device)]

    results = []
    for i in range(batch_sz):
        s, e = sample_ranges[i]
        if not sample_valid[i]:
            results.append((False, ctc_texts[i]))
            continue
        token_probs = ctc_probs[s:e]
        accepted    = (token_probs >= CONFIDENCE_THRESHOLD).all().item()
        results.append((accepted, ctc_texts[i]))

    return results, audio_embeds, proj_lengths

## Step 3: LLM Verification

The BPE draft tokens are already in the LLM's vocabulary, so verification is straightforward:
build `[prefix | audio_embeds | suffix | draft_tokens]`, run one LLM forward pass, and
check that the model's probability for each draft token exceeds `CONFIDENCE_THRESHOLD`.

The `enc_hidden` from the CTC step is projected through `model.projector` here — no second
encoder pass is needed.

In [ ]:
bpe_texts, bpe_entropies, embeddings, embed_lengths = ctc_decode_bpe([audio])

print(f"BPE CTC text:    {bpe_texts[0]}")
print(f"BPE entropy (max): {bpe_entropies[0]:.4f}")
print(f"enc_hidden shape:  {embeddings.shape}")
print(f"\nCTC threshold: {CTC_THRESHOLD}")
print(f"Accept directly?  {bpe_entropies[0] <= CTC_THRESHOLD}")

In [ ]:
@torch.no_grad()
def ctc_decode_bpe(audios):
    """BPE CTC draft: single encoder pass, reuse embeddings for verify/fallback.

    Returns:
        bpe_texts:    List of decoded strings
        bpe_entropies: List of max entropy values (confidence measure)
        enc_hidden:   Full-resolution encoder hidden states [B, T_enc, D]
                      (passed unchanged to model.projector for verify/fallback)
        embed_lengths: Unpadded encoder frame count per sample
    """
    texts = [text_prompt] * len(audios)
    model_inputs = processor(texts, audios, device=device, return_tensors="pt").to(device)

    autocast = (torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
                if device.startswith('cuda') else contextlib.nullcontext())

    with autocast:
        encoder_output = model.encoder(model_inputs["input_features"])
        enc_hidden = (encoder_output.last_hidden_state
                      if hasattr(encoder_output, 'last_hidden_state')
                      else encoder_output)

        # Mid-layer grapheme logits for importance weighting (hook populated _mid_hidden)
        mid_h = _mid_hidden['h']
        mid_grapheme_logits = model.encoder.out(mid_h)
        grapheme_probs_mid  = F.softmax(mid_grapheme_logits.float(), dim=-1)
        importance = 1.0 - grapheme_probs_mid[:, :, 0]  # 1 - blank_prob

        # 4x pooling for BPE head — enc_hidden itself is never modified
        x_pooled, _ = posterior_weighted_pool(enc_hidden, importance,
                                               window_size=LLM_DOWNSAMPLE_WINDOW)

        # Valid pooled lengths per sample
        pooled_lengths = []
        for i in range(len(audios)):
            enc_len = min(len(audios[i]) // HOP_LENGTH // 2 + 1, enc_hidden.shape[1])
            pooled_len = math.ceil(enc_len / LLM_DOWNSAMPLE_WINDOW)
            pooled_lengths.append(min(pooled_len, x_pooled.shape[1]))

        # Gather valid frames across batch and apply BPE head
        valid_positions = [(i, t) for i, plen in enumerate(pooled_lengths) for t in range(plen)]
        batch_idx = torch.tensor([p[0] for p in valid_positions], device=device)
        time_idx  = torch.tensor([p[1] for p in valid_positions], device=device)
        x_valid          = x_pooled[batch_idx, time_idx, :]   # [N_valid, D]
        bpe_logits_valid = out_llm(x_valid)                    # [N_valid, LLM_OUT_DIM]
        bpe_probs_valid  = F.softmax(bpe_logits_valid.float(), dim=-1)

    # Decode each sample
    bpe_texts, bpe_entropies, embed_lengths = [], [], []
    offset = 0
    for i in range(len(audios)):
        plen    = pooled_lengths[i]
        probs_i = bpe_probs_valid[offset:offset + plen]
        offset += plen

        _, idx      = probs_i.max(dim=-1)
        entropy_i   = -(probs_i * torch.log(probs_i + 1e-10)).sum(dim=-1)
        dedup       = torch.unique_consecutive(idx, dim=-1)
        non_blank   = dedup[dedup > 0]
        token_ids   = [t.item() - 1 for t in non_blank]  # label i -> Granite token (i-1)
        text        = tokenizer.decode(token_ids) if token_ids else ""

        bpe_texts.append(text)
        bpe_entropies.append(entropy_i.max().item() if token_ids else float('inf'))
        embed_lengths.append(len(audios[i]) // HOP_LENGTH // 2 + 1)

    return bpe_texts, bpe_entropies, enc_hidden, embed_lengths

## Step 2: BPE CTC Decoding

Key differences from the grapheme CTC notebook:

- **Single encoder pass** — `enc_hidden` is captured and reused for verify/fallback
- **Importance-weighted pooling** reduces `enc_hidden` 4× before the BPE head
- **Token mapping** — BPE label `i` → Granite tokenizer token `i-1` (blank occupies label 0)
- **Entropy** is computed over the BPE vocabulary (~100K); thresholds differ from the grapheme case

In [ ]:
def posterior_weighted_pool(hidden, importance, window_size=4):
    """Importance-weighted downsampling.

    Args:
        hidden:     [B, T, D] encoder hidden states
        importance: [B, T]    per-frame weights (1 - blank_prob)
        window_size: temporal downsampling factor
    Returns:
        pooled: [B, T//window_size, D]
        window_size (pass-through for callers)
    """
    B, T, D = hidden.shape
    pad_len = (window_size - T % window_size) % window_size
    if pad_len > 0:
        hidden     = F.pad(hidden,     (0, 0, 0, pad_len))
        importance = F.pad(importance, (0, pad_len))
    num_windows = hidden.shape[1] // window_size
    hidden     = hidden.view(B, num_windows, window_size, D)
    importance = importance.view(B, num_windows, window_size)
    weights = importance / (importance.sum(dim=-1, keepdim=True) + 1e-8)
    return (hidden * weights.unsqueeze(-1)).sum(dim=2), window_size

## Step 1: Importance-Weighted Pooling

Before applying the BPE CTC head, the encoder hidden states are downsampled 4x using
importance weights derived from the **mid-layer grapheme blank probability**.

Frames where the grapheme model assigns high blank probability (silence/noise) get low
weight; speech frames get high weight.  This concentrates semantic content in fewer
vectors before the BPE linear projection.

In [ ]:
audio_path = hf_hub_download(repo_id=MODEL_ID, filename="multilingual_sample.wav")
audio, sample_rate = sf.read(audio_path)

if sample_rate != 16000:
    audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=16000)

print(f"Audio: {audio_path}")
print(f"Length: {len(audio) / 16000:.2f}s")
ipd.Audio(audio, rate=16000)

## Load Sample Audio

In [ ]:
# Speculative decoding thresholds
CONFIDENCE_THRESHOLD = 0.2   # LLM verification acceptance threshold (per token)
CTC_THRESHOLD        = 0.7   # BPE CTC entropy threshold for direct acceptance

# Generation parameters
MAX_NEW_TOKENS = 200
NUM_BEAMS      = 1

# Audio parameters
HOP_LENGTH = 160  # samples per frame (10ms at 16kHz)

## Configuration

In [ ]:
text_instruction = "<|audio|>can you transcribe the speech into a written format?"
message = [{"role": "user", "content": text_instruction}]
text_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

prompt_prefix, prompt_suffix = text_prompt.split("<|audio|>")
print(f"Prefix: {repr(prompt_prefix)}")
print(f"Suffix: {repr(prompt_suffix)}")

embed_layer = model.language_model.get_input_embeddings()
prefix_ids = tokenizer.encode(prompt_prefix, add_special_tokens=False)
suffix_ids  = tokenizer.encode(prompt_suffix,  add_special_tokens=False)
cached_prefix_embeds = embed_layer(torch.tensor([prefix_ids], device=device))
cached_suffix_embeds = embed_layer(torch.tensor([suffix_ids],  device=device))

print(f"\nPrefix tokens: {len(prefix_ids)}, Suffix tokens: {len(suffix_ids)}")
print(f"Cached prefix embeds: {cached_prefix_embeds.shape}")
print(f"Cached suffix embeds: {cached_suffix_embeds.shape}")

## Setup Chat Template and Cache Prompt Embeddings

In [ ]:
LLM_DOWNSAMPLE_WINDOW = 4
LLM_OUT_DIM = 100353  # 100352 Granite BPE tokens + 1 CTC blank (label 0)

# Load BPE CTC head weights from the model repo
out_llm_path = hf_hub_download(repo_id=MODEL_ID, filename="out_llm.safetensors")
out_llm = nn.Linear(model.encoder.config.hidden_dim, LLM_OUT_DIM, bias=True)
out_llm.load_state_dict(load_file(out_llm_path))
out_llm.to(dtype).eval().to(device)
print(f"BPE CTC head loaded: Linear({model.encoder.config.hidden_dim}, {LLM_OUT_DIM})")

# Forward hook: capture mid-layer hidden state for importance weighting.
# The grapheme feedback is applied at layer num_layers//2; we hook just before it
# to compute blank probability from model.encoder.out (grapheme head).
num_enc_layers = model.encoder.config.num_layers
mid_layer_idx = num_enc_layers // 2 - 1
_mid_hidden = {}

def _save_mid_hidden(module, input, output):
    _mid_hidden['h'] = output[0] if isinstance(output, tuple) else output

_hook = model.encoder.layers[mid_layer_idx].register_forward_hook(_save_mid_hidden)
print(f"Registered mid-layer hook at encoder layer {mid_layer_idx} (of {num_enc_layers})")

## Load BPE CTC Head and Register Mid-Layer Hook

The dual-head encoder keeps the standard Granite Speech encoder weights unchanged.
Only the BPE linear projection (`out_llm`) is new — it maps encoder hidden states to
the LLM's BPE vocabulary (100352 tokens + 1 CTC blank = 100353 labels).

The mid-layer hook captures the grapheme blank probability at layer `num_layers//2`,
used to compute importance weights for the 4x temporal pooling step.

In [ ]:
MODEL_ID = "ibm-granite/granite-speech-4.1-2b"

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

dtype = torch.bfloat16 if device == "cuda" else torch.float32
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()

logits_scaling = getattr(model.language_model.config, 'logits_scaling', 1.0)
print(f"Model loaded on {device} with dtype {dtype}. Logits scaling: {logits_scaling}")

## Load Model and Processor

In [ ]:
import contextlib
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, models
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import soundfile as sf
import librosa
import IPython.display as ipd

assert hasattr(models, "granite_speech"), "Please install transformers with granite_speech support"

torch.set_float32_matmul_precision('high')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: Running on CPU. bfloat16 autocast is disabled; expect slow inference.")

## Setup

# Self-Speculative Decoding with Dual Graphemic/BPE CTC Encoder

This notebook demonstrates **self-speculative decoding** using the **dual-head CTC encoder** for [granite-speech-4.1-2b](https://huggingface.co/ibm-granite/granite-speech-4.1-2b).

Compared to the graphemic CTC notebook, the BPE CTC head drafts tokens that live directly in the LLM's vocabulary, making verification cheaper and more accurate:

1. **BPE CTC encoder** (`out_llm` head loaded from the model repo) generates draft token sequences via importance-weighted pooling
2. **LLM verifies** the draft in a single forward pass
3. If verification fails, **AR fallback** produces the final transcription

The encoder is run only once; the resulting hidden states are reused across all three steps.